# 🏥 Análise de Dados — SIA/SUS Produção Ambulatorial
## Por Local de Atendimento | Jan/2024 a Jan/2026

**Fonte:** DATASUS — TabNet / SIA/SUS  
**Abrangência:** Brasil (5.600 municípios)  
**Conteúdo:** Quantidade Aprovada e Valor Aprovado  
**Coluna:** Subgrupo de Procedimento (51 subgrupos)  
**Períodos:** 25 meses (Jan/2024 → Jan/2026)

---

## 1. Dependências

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)

plt.rcParams.update({
    'figure.dpi'     : 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.family'    : 'DejaVu Sans',
})

DADOS_DIR = Path('dados')
print(f'Arquivos CSV encontrados: {len(list(DADOS_DIR.glob("*.csv")))}')

## 2. Carga e Exploração dos Dados

In [ ]:
def carregar_csvs(tipo: str) -> pd.DataFrame:
    """Carrega e concatena todos os CSVs de um tipo (qtd_aprovada ou valor_aprovado)."""
    arquivos = sorted(DADOS_DIR.glob(f'sia_{tipo}_*.csv'))
    frames = []
    for arq in arquivos:
        df = pd.read_csv(arq, sep=';', encoding='utf-8-sig', dtype=str)
        frames.append(df)
    return pd.concat(frames, ignore_index=True)

df_qtd   = carregar_csvs('qtd_aprovada')
df_valor = carregar_csvs('valor_aprovado')

print('=== Qtd. Aprovada ===')
print(f'Shape: {df_qtd.shape}')
print(df_qtd.head(3))
print()
print('=== Valor Aprovado ===')
print(f'Shape: {df_valor.shape}')
print(df_valor.head(3))

## 3. Limpeza dos Dados

In [ ]:
COLUNAS_FIXAS = ['periodo', 'conteudo', 'Município']

def limpar(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Normaliza nome da coluna município
    df.columns = [c.strip().strip('"') for c in df.columns]
    col_mun = [c for c in df.columns if 'unic' in c.lower()][0]
    df.rename(columns={col_mun: 'municipio'}, inplace=True)

    # Extrai código e nome do município
    df['cod_municipio'] = df['municipio'].str.extract(r'^(\d{6})')
    df['municipio']     = df['municipio'].str.replace(r'^\d{6}\s*', '', regex=True).str.strip()

    # Converte período para datetime
    df['periodo_dt'] = pd.to_datetime(df['periodo'], format='%m/%Y')
    df['ano']        = df['periodo_dt'].dt.year
    df['mes']        = df['periodo_dt'].dt.month

    # Colunas de subgrupo = tudo exceto fixas e Total
    subgrupos = [c for c in df.columns
                 if c not in ('periodo','conteudo','municipio','cod_municipio','periodo_dt','ano','mes','Total')]

    # Converte "-" e valores numéricos (formato brasileiro: ponto=milhar, vírgula=decimal)
    for col in subgrupos + ['Total']:
        if col in df.columns:
            df[col] = (df[col].astype(str)
                       .str.strip().str.strip('"')
                       .replace('-', '0')
                       .str.replace('.', '', regex=False)
                       .str.replace(',', '.', regex=False))
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

    # Remove linha "Total" do município (linha de rodapé do TabNet)
    df = df[~df['municipio'].str.upper().str.contains('^TOTAL$', na=False)]

    return df

df_qtd   = limpar(df_qtd)
df_valor = limpar(df_valor)

# Identifica subgrupos
SUBGRUPOS = [c for c in df_qtd.columns
             if c not in ('periodo','conteudo','municipio','cod_municipio','periodo_dt','ano','mes','Total')]

print(f'Linhas após limpeza — Qtd: {len(df_qtd):,} | Valor: {len(df_valor):,}')
print(f'Subgrupos de procedimento: {len(SUBGRUPOS)}')
print(f'Municípios únicos: {df_qtd["municipio"].nunique():,}')
print(f'Períodos: {sorted(df_qtd["periodo"].unique())}')
print()
print('Valores nulos por coluna (primeiras 5):')
print(df_qtd[SUBGRUPOS[:5]].isnull().sum())

## 4. Estatísticas Descritivas

In [ ]:
# Resumo por período
resumo_periodo = (
    df_qtd.groupby('periodo_dt')['Total'].sum()
    .reset_index()
    .rename(columns={'Total': 'qtd_total'})
    .merge(
        df_valor.groupby('periodo_dt')['Total'].sum().reset_index().rename(columns={'Total': 'valor_total'}),
        on='periodo_dt'
    )
)
resumo_periodo['periodo'] = resumo_periodo['periodo_dt'].dt.strftime('%m/%Y')
resumo_periodo = resumo_periodo.sort_values('periodo_dt')

print('=== Resumo por Período ===')
print(resumo_periodo[['periodo','qtd_total','valor_total']].to_string(index=False))
print()
print('=== Estatísticas da Qtd. Total Mensal ===')
print(resumo_periodo['qtd_total'].describe().apply(lambda x: f'{x:,.0f}'))

## 5. Visualizações

### 5.1 Série Temporal — Qtd. Aprovada e Valor Aprovado

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
fig.suptitle('SIA/SUS — Produção Ambulatorial — Brasil\nJan/2024 a Jan/2026', fontsize=14, fontweight='bold')

# Qtd. Aprovada
ax1.bar(resumo_periodo['periodo_dt'], resumo_periodo['qtd_total'] / 1e6,
        color='steelblue', alpha=0.8, width=20)
ax1.plot(resumo_periodo['periodo_dt'], resumo_periodo['qtd_total'] / 1e6,
         color='navy', linewidth=1.5, marker='o', markersize=4)
ax1.set_ylabel('Qtd. Aprovada (milhões)', fontsize=10)
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}M'))
ax1.set_title('Quantidade Aprovada', fontsize=11)
ax1.grid(axis='y', alpha=0.3)

# Valor Aprovado
ax2.bar(resumo_periodo['periodo_dt'], resumo_periodo['valor_total'] / 1e9,
        color='seagreen', alpha=0.8, width=20)
ax2.plot(resumo_periodo['periodo_dt'], resumo_periodo['valor_total'] / 1e9,
         color='darkgreen', linewidth=1.5, marker='o', markersize=4)
ax2.set_ylabel('Valor Aprovado (R$ bilhões)', fontsize=10)
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'R${x:,.1f}Bi'))
ax2.set_title('Valor Aprovado', fontsize=11)
ax2.grid(axis='y', alpha=0.3)

plt.xticks(rotation=45, ha='right', fontsize=8)
plt.tight_layout()
plt.savefig('serie_temporal.png', bbox_inches='tight')
plt.show()

### 5.2 Top 10 Subgrupos de Procedimento

In [ ]:
top_sub_qtd = (
    df_qtd[SUBGRUPOS].sum()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
    .rename(columns={'index': 'subgrupo', 0: 'qtd'})
)
top_sub_qtd.columns = ['subgrupo', 'qtd']
top_sub_qtd['subgrupo_curto'] = top_sub_qtd['subgrupo'].str.slice(0, 40)
top_sub_qtd['pct'] = top_sub_qtd['qtd'] / top_sub_qtd['qtd'].sum() * 100

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Top 10 Subgrupos de Procedimento — Qtd. Aprovada (Jan/2024–Jan/2026)',
             fontsize=13, fontweight='bold')

# Barras horizontais
colors = sns.color_palette('Blues_r', 10)
bars = ax1.barh(top_sub_qtd['subgrupo_curto'][::-1], top_sub_qtd['qtd'][::-1] / 1e9, color=colors)
ax1.set_xlabel('Qtd. Aprovada (bilhões)')
ax1.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.1f}Bi'))
for bar, val in zip(bars, top_sub_qtd['qtd'][::-1]):
    ax1.text(bar.get_width() + 0.02, bar.get_y() + bar.get_height()/2,
             f'{val/1e9:,.1f}Bi', va='center', fontsize=8)
ax1.grid(axis='x', alpha=0.3)

# Pizza
wedge_colors = sns.color_palette('Blues', 10)
wedges, texts, autotexts = ax2.pie(
    top_sub_qtd['pct'],
    labels=[s[:25] for s in top_sub_qtd['subgrupo_curto']],
    autopct='%1.1f%%',
    colors=wedge_colors,
    startangle=140,
    pctdistance=0.75,
    textprops={'fontsize': 7}
)
ax2.set_title('Participação % — Top 10', fontsize=11)

plt.tight_layout()
plt.savefig('top10_subgrupos.png', bbox_inches='tight')
plt.show()

print(top_sub_qtd[['subgrupo','qtd','pct']].to_string(index=False))

### 5.3 Top 15 Municípios por Qtd. Aprovada

In [ ]:
top_mun = (
    df_qtd[df_qtd['municipio'].str.strip() != '']
    .groupby('municipio')['Total']
    .sum()
    .sort_values(ascending=False)
    .head(15)
    .reset_index()
)
top_mun.columns = ['municipio', 'qtd_total']
top_mun['municipio_curto'] = top_mun['municipio'].str.slice(0, 30)
top_mun['pct'] = top_mun['qtd_total'] / df_qtd['Total'].sum() * 100

fig, ax = plt.subplots(figsize=(12, 7))
colors = sns.color_palette('viridis', 15)
bars = ax.barh(top_mun['municipio_curto'][::-1], top_mun['qtd_total'][::-1] / 1e6, color=colors[::-1])

for bar, row in zip(bars, top_mun.iloc[::-1].itertuples()):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
            f'{row.qtd_total/1e6:,.1f}M  ({row.pct:.2f}%)', va='center', fontsize=8)

ax.set_xlabel('Qtd. Aprovada (milhões)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}M'))
ax.set_title('Top 15 Municípios — Qtd. Aprovada\nJan/2024 a Jan/2026', fontsize=13, fontweight='bold')
ax.grid(axis='x', alpha=0.3)
ax.set_xlim(0, top_mun['qtd_total'].max() / 1e6 * 1.25)

plt.tight_layout()
plt.savefig('top15_municipios.png', bbox_inches='tight')
plt.show()

### 5.4 Heatmap — Sazonalidade por Subgrupo (Top 10)

In [ ]:
top10_subs = top_sub_qtd['subgrupo'].tolist()

heat_data = (
    df_qtd.groupby('periodo_dt')[top10_subs].sum()
    .sort_index()
)
heat_data.index = heat_data.index.strftime('%m/%Y')
# Normaliza por coluna para mostrar variação relativa
heat_norm = heat_data.div(heat_data.mean()).T
heat_norm.index = [s[:35] for s in heat_norm.index]

fig, ax = plt.subplots(figsize=(16, 7))
sns.heatmap(heat_norm, ax=ax, cmap='YlOrRd', linewidths=0.3,
            annot=False, cbar_kws={'label': 'Índice (1 = média)'})
ax.set_title('Sazonalidade por Subgrupo de Procedimento\n(valores normalizados pela média do período)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Período')
ax.set_ylabel('')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(fontsize=8)
plt.tight_layout()
plt.savefig('heatmap_sazonalidade.png', bbox_inches='tight')
plt.show()

### 5.5 Variação Anual 2024 vs 2025

In [ ]:
anual_qtd = df_qtd.groupby('ano')['Total'].sum()
anual_val = df_valor.groupby('ano')['Total'].sum()

# Crescimento por subgrupo: 2024 vs 2025
sub_2024 = df_qtd[df_qtd['ano'] == 2024][SUBGRUPOS].sum()
sub_2025 = df_qtd[df_qtd['ano'] == 2025][SUBGRUPOS].sum()
cresc_sub = ((sub_2025 - sub_2024) / sub_2024.replace(0, np.nan) * 100).dropna().sort_values()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Variação 2024 → 2025', fontsize=14, fontweight='bold')

# Crescimento total
anos_qtd = anual_qtd[anual_qtd.index.isin([2024, 2025])]
anos_val = anual_val[anual_val.index.isin([2024, 2025])]
x = np.arange(2)
width = 0.35
axes[0].bar(x - width/2, anos_qtd.values / 1e9, width, label='Qtd (bilhões)', color='steelblue', alpha=0.8)
axes[0].bar(x + width/2, anos_val.values / 1e9,  width, label='Valor (R$ Bi)', color='seagreen', alpha=0.8)
axes[0].set_xticks(x)
axes[0].set_xticklabels(['2024', '2025'])
axes[0].set_title('Total Anual', fontsize=11)
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)
cresc_pct = ((anos_qtd.iloc[1] - anos_qtd.iloc[0]) / anos_qtd.iloc[0]) * 100
axes[0].text(0.5, 0.95, f'Crescimento Qtd: +{cresc_pct:.1f}%',
             transform=axes[0].transAxes, ha='center', fontsize=11,
             color='navy', fontweight='bold')

# Crescimento por subgrupo (top e bottom 8)
extremos = pd.concat([cresc_sub.head(8), cresc_sub.tail(8)])
colors_c  = ['tomato' if v < 0 else 'steelblue' for v in extremos.values]
axes[1].barh([s[:35] for s in extremos.index], extremos.values, color=colors_c, alpha=0.8)
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_xlabel('Variação %')
axes[1].set_title('Maiores Altas e Quedas por Subgrupo', fontsize=11)
axes[1].grid(axis='x', alpha=0.3)
plt.yticks(fontsize=7)

plt.tight_layout()
plt.savefig('variacao_anual.png', bbox_inches='tight')
plt.show()

### 5.6 Distribuição de Municípios por Faixa de Produção

In [ ]:
prod_mun = (
    df_qtd[df_qtd['municipio'].str.strip() != '']
    .groupby('municipio')['Total'].sum()
)

faixas = pd.cut(prod_mun,
    bins=[0, 1_000, 100_000, 1_000_000, 10_000_000, 100_000_000, float('inf')],
    labels=['< 1K', '1K–100K', '100K–1M', '1M–10M', '10M–100M', '> 100M']
)
dist = faixas.value_counts().sort_index()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Distribuição de Municípios por Volume de Produção\n(Qtd. Aprovada acumulada Jan/2024–Jan/2026)',
             fontsize=13, fontweight='bold')

# Barras
colors = sns.color_palette('Blues', len(dist))
bars = ax1.bar(dist.index, dist.values, color=colors, edgecolor='white')
for bar, v in zip(bars, dist.values):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
             f'{v:,}\n({v/len(prod_mun)*100:.1f}%)', ha='center', fontsize=9)
ax1.set_xlabel('Faixa de Produção')
ax1.set_ylabel('Nº de Municípios')
ax1.set_title('Número de Municípios por Faixa')
ax1.grid(axis='y', alpha=0.3)

# Boxplot (log scale)
ax2.boxplot(prod_mun[prod_mun > 0], vert=True, patch_artist=True,
            boxprops=dict(facecolor='steelblue', alpha=0.6))
ax2.set_yscale('log')
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax2.set_title('Distribuição (escala log)')
ax2.set_ylabel('Qtd. Aprovada Total (log)')
ax2.grid(axis='y', alpha=0.3)

# Percentis
p = np.percentile(prod_mun[prod_mun > 0], [25, 50, 75, 90, 99])
for pv, label in zip(p, ['P25','P50','P75','P90','P99']):
    ax2.axhline(pv, linestyle='--', alpha=0.6, linewidth=0.8)
    ax2.text(1.02, pv, f'{label}: {pv:,.0f}', va='center', fontsize=7, transform=ax2.get_yaxis_transform())

plt.tight_layout()
plt.savefig('distribuicao_municipios.png', bbox_inches='tight')
plt.show()

## 6. Principais Insights e Recomendações

In [ ]:
total_qtd   = df_qtd['Total'].sum()
total_valor = df_valor['Total'].sum()
municipios  = df_qtd['municipio'].nunique()
cresc_2025  = ((anual_qtd.get(2025,0) - anual_qtd.get(2024,0)) / anual_qtd.get(2024,1)) * 100
zeradas_pct = (df_qtd['Total'] == 0).mean() * 100
top1_sub    = top_sub_qtd.iloc[0]
top1_mun    = top_mun.iloc[0]

print('=' * 65)
print('  RELATÓRIO DE ANÁLISE — SIA/SUS Produção Ambulatorial')
print('  Brasil | Jan/2024 a Jan/2026')
print('=' * 65)

print(f"""
1. VISÃO GERAL DO DATASET
   ├─ Arquivos CSV      : 50  (25 meses × 2 conteúdos)
   ├─ Municípios        : {municipios:,}
   ├─ Subgrupos proced. : {len(SUBGRUPOS)}
   ├─ Qtd. Aprov. Total : {total_qtd/1e9:,.2f} bilhões
   └─ Valor Aprov. Total: R$ {total_valor/1e9:,.2f} bilhões

2. QUALIDADE DOS DADOS
   ├─ Linhas zeradas    : {zeradas_pct:.1f}% (baixo — dados bem preenchidos)
   └─ Valores nulos     : 0 (tratados na limpeza)

3. TENDÊNCIA
   ├─ Crescimento 2024→2025 : +{cresc_2025:.1f}%
   ├─ Pico de produção      : Out/2025 (~929 milhões)
   └─ Sazonalidade          : Quedas em Dez/Jan (férias/feriados)

4. CONCENTRAÇÃO DE PROCEDIMENTOS
   ├─ Top subgrupo  : {top1_sub['subgrupo'][:50]}
   │                  {top1_sub['qtd']/1e9:.1f}Bi proc. ({top1_sub['pct']:.1f}% do total)
   └─ Consultas+Lab : ~51,7% de toda a produção ambulatorial

5. CONCENTRAÇÃO GEOGRÁFICA
   ├─ Top município : {top1_mun['municipio']} ({top1_mun['pct']:.2f}% do total)
   └─ Distribuição  : relativamente descentralizada (5.600 municípios ativos)

6. RECOMENDAÇÕES
   1. Monitorar subgrupos com queda em 2025 para identificar
      possíveis problemas de acesso ou financiamento
   2. Investigar municípios com produção zero em múltiplos meses
      (possível subnotificação ou ausência de serviços)
   3. Analisar sazonalidade de Dez/Jan para planejamento de capacidade
   4. Cruzar dados com população IBGE para calcular cobertura per capita
""")
print('=' * 65)